In [ ]:
# Load the dataset
data_file = '/content/WineQT.csv'
data = pd.read_csv(data_file)



In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Inspect the dataset
data.info()
data.describe()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000
mean,8.311111,0.531339,0.268364,2.532152,0.086933,15.615486,45.914698,0.996730,3.311015,0.657708,10.442111,5.657043,804.969379
std,1.747595,0.179633,0.196686,1.355917,0.047267,10.250486,32.782130,0.001925,0.156664,0.170399,1.082196,0.805824,463.997116
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000,0.000000
25%,7.100000,0.392500,0.090000,1.900000,0.070000,7.000000,21.000000,0.995570,3.205000,0.550000,9.500000,5.000000,411.000000
50%,7.900000,0.520000,0.250000,2.200000,0.079000,13.000000,37.000000,0.996680,3.310000,0.620000,10.200000,6.000000,794.000000
75%,9.100000,0.640000,0.420000,2.600000,0.090000,21.000000,61.000000,0.997845,3.400000,0.730000,11.100000,6.000000,1209.500000
max,15.900000,1.580000,1.000000,15.500000,0.611000,68.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000,1597.000000


In [ ]:
# Add a new column to categorize wine quality
data['quality_label'] = data['quality'].apply(lambda score: 'Good' if score >= 6 else 'Bad')



In [ ]:
# Identify numerical features for the model (exclude non-numeric columns and target)
feature_columns = data.drop(columns=['quality', 'quality_label']).select_dtypes(include='number').columns.tolist()
X_features = data[feature_columns]
y_target = data['quality_label']



In [ ]:
# Split the dataset into training and testing sets (75% train, 25% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_features, y_target, test_size=0.25, random_state=42, stratify=y_target
)



In [ ]:
# Naive Bayes Classifier (from scratch)
class SimpleNaiveBayes:
    def __init__(self):
        self.class_probs = {}
        self.feature_info = {}

    def fit(self, X_train, y_train):
        # Get unique classes
        classes = np.unique(y_train)
        for cls in classes:
            # Filter rows for the current class
            X_cls = X_train[y_train == cls]
            # Calculate class probability
            self.class_probs[cls] = len(X_cls) / len(X_train)
            # Calculate mean and variance for each feature
            self.feature_info[cls] = {
                'mean': X_cls.mean(axis=0).values,
                'var': X_cls.var(axis=0).values
            }

    def calc_probability(self, x, mean, var):
        # Use Gaussian formula
        exp = np.exp(-((x - mean) ** 2) / (2 * var))
        return (1 / np.sqrt(2 * np.pi * var)) * exp

    def predict(self, X_test):
        predictions = []
        for _, row in X_test.iterrows():
            class_scores = {}
            for cls, stats in self.feature_info.items():
                # Start with class probability
                class_scores[cls] = np.log(self.class_probs[cls])
                # Add log of feature probabilities
                feature_probs = self.calc_probability(
                    row.values, stats['mean'], stats['var']
                )
                class_scores[cls] += np.sum(np.log(feature_probs))
            # Choose class with highest score
            predictions.append(max(class_scores, key=class_scores.get))
        return predictions



In [ ]:
# Make predictions on the test set
predicted_labels = classifier.predict(X_test)



In [ ]:
# Evaluate the model's performance
model_accuracy = accuracy_score(y_test, predicted_labels)
report = classification_report(y_test, predicted_labels)



In [ ]:
# Output the results
print(f"Model Accuracy: {model_accuracy:.2f}")
print("\nDetailed Classification Report:")
print(report)


Model Accuracy: 0.76

Detailed Classification Report:
              precision    recall  f1-score   support

         Bad       0.73      0.73      0.73       131
        Good       0.77      0.77      0.77       155

    accuracy                           0.75       286
   macro avg       0.75      0.75      0.75       286
weighted avg       0.75      0.75      0.75       286

